# Neutrino Pile-up Study -- ICARUS BNB MC

**Author:** Sparshita Dey  
**Last modified:** May $2026$

---

## Motivation

This notebook addresses a question raised during a collaboration discussion:

> *"If someone could do a truth-level dump of the main ICARUS MC sample and comment on how often there is more than one neutrino that leaves EDeps in the detectors -- that is the main figure of merit. If that number is non-negligible then we should perhaps try to do it SBND-style; otherwise it is probably overkill."*

A related quick check was also performed by a collaborator:

> *"Look at `nu_id` in the CAF files for some $1\mu$Np signal events and see what fraction of the time it is $1$ and not $0$. The answer is $0.0099$, so pretty much $1$%."*

This was followed by the natural question: does that number include **all** neutrinos generated anywhere in the volume, or only ones that actually interact in the detector?

We address both questions systematically:

| Study | Method | What it measures |
|---|---|---|
| **A -- Quick proxy** | `truth.index` of reconstructed slices | Fraction of reconstructed slices matched to $2$nd or later neutrino (FV only, fast) |
| **B -- All neutrinos in spill** | `rec.mc.nnu` per spill | All neutrinos saved in CAF per spill (upper bound) |
| **C -- FV neutrinos in spill** | Vertex position cuts | Neutrinos with vertex in fiducial volume per spill (**main FOM**) |

The notebook is split into:
- **Part 1:** Pilot on $200$ files (~$10{,}000$ spills) -- runs inline, fast
- **Part 2:** Full run over all files using a chunked approach. Files are split into chunks of $1{,}000$. Each chunk saves its result to disk immediately, so if a terminal hangs or is interrupted, the run can be resumed from where it left off without losing any work. Multiple terminals can process different chunk ranges in parallel to reduce wall-clock time.

---

## $0$. Setup

In [ ]:
import uproot
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import warnings
warnings.filterwarnings('ignore')

# unicode symbols for print statements
PM = '\u00b1'

# matplotlib named colours -- no hex
C_ALL      = 'powderblue'
C_FV       = 'black'
C_REF_ALL  = 'steelblue'
C_REF_FV   = 'crimson'
C_ANNO     = 'dimgrey'
C_EDGE     = 'dimgrey'

plt.rcParams.update({
    'font.family':         'serif',
    'font.serif':          ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset':    'cm',
    'axes.labelsize':      13,
    'axes.titlesize':      13,
    'axes.titlepad':       10,
    'xtick.labelsize':     11,
    'ytick.labelsize':     11,
    'legend.fontsize':     9,
    'legend.framealpha':   0.92,
    'legend.edgecolor':    'lightgrey',
    'legend.fancybox':     True,
    'figure.dpi':          150,
    'figure.facecolor':    'white',
    'axes.facecolor':      'white',
    'axes.edgecolor':      'dimgrey',
    'axes.linewidth':      1.2,
    'axes.grid':           False,
    'xtick.direction':     'in',
    'ytick.direction':     'in',
    'xtick.top':           True,
    'ytick.right':         True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
})

# ── file list ─────────────────────────────────────────────────────────────────
FILELIST   = "/exp/icarus/app/users/sdey2/fileLists/BNBflatcafs_v10_06_00_06p03_Overlays_All.list"
CHUNK_DIR  = "/exp/icarus/app/users/sdey2/icaruscode-v10_06_00_04p04_production/nugraphProduction/localRunTest/pileup_chunks"   # where chunk results are saved
CHUNK_SIZE = 1000
PRINT_EVERY = 10

with open(FILELIST) as fh:
    ALL_FILES = [line.strip() for line in fh if line.strip()]
os.makedirs(CHUNK_DIR, exist_ok=True)

# split into chunks
CHUNKS = [ALL_FILES[i:i+CHUNK_SIZE]
          for i in range(0, len(ALL_FILES), CHUNK_SIZE)]
print(f"Total files   : {len(ALL_FILES):,}")
print(f"Chunk size    : {CHUNK_SIZE}")
print(f"Total chunks  : {len(CHUNKS)}")
print(f"Chunk save dir: {CHUNK_DIR}")

# ── ICARUS FV definition -- matches Config.h ──────────────────────────────────
# verified: rec.mc.nu.position.x/y/z are in the correct coordinate frame
def in_fv(x, y, z, mx=25, my=25, mz_up=30, mz_dn=50):
    in_tpc = (((x < -61.94 - mx) & (x > -358.49 + mx)) |
              ((x >  61.94 + mx) & (x <  358.49 - mx)))
    in_y   = (y > -181.86 + my) & (y < 134.96  - my)
    in_z   = (z > -894.95 + mz_up) & (z < 894.95 - mz_dn)
    return in_tpc & in_y & in_z

def open_tree(fpath):
    """Open a flat CAF and return the correct tree regardless of name."""
    try:
        f = uproot.open(fpath)
        for name in ['recTree', 'globalTree']:
            if name in f:
                return f[name]
    except Exception:
        pass
    return None

def binom_err(n_pass, n_total):
    """Binomial standard error on a proportion."""
    if n_total == 0:
        return 0
    p = n_pass / n_total
    return np.sqrt(p * (1 - p) / n_total)

def make_combined_plot(ax, arr_all, arr_fv, title, is_pilot=True):
    """Side-by-side bars for Study B (all nu) and Study C (FV nu).
    Uncertainty shown as hatched rectangles spanning +/- 1 sigma.
    """
    N     = len(arr_all)
    xs    = np.array([0, 1, 2, 3])
    w     = 0.38
    xlbls = ['$0$', '$1$', '$2$', r'$\geq 3$']

    studies = [
        (arr_all, -w/2, C_ALL,   r'All $\nu$ (Study B -- upper bound)'),
        (arr_fv,  +w/2, C_FV,    r'FV $\nu$ (Study C -- figure of merit)'),
    ]

    for arr, offset, colour, lbl in studies:
        counts = [(arr == k).sum() if k < 3 else (arr >= 3).sum()
                  for k in range(4)]
        pcts   = [100*c/N for c in counts]
        errs   = [100*binom_err(c, N) for c in counts]

        ax.bar(xs + offset, pcts, width=w,
               color=colour,
               edgecolor=C_EDGE, linewidth=0.8,
               label=lbl, zorder=3)

        # hatched uncertainty rectangles
        for x, pct, err in zip(xs, pcts, errs):
            if pct > 0 and err > 0:
                rect = plt.Rectangle(
                    (x + offset - w/2, pct - err),
                    w, 2*err,
                    hatch='xxx', fill=False,
                    edgecolor=C_ANNO, linewidth=0.4, zorder=5)
                ax.add_patch(rect)

        # value labels on pile-up bins only
        for x, pct, err in zip(xs, pcts, errs):
            if x >= 2 and pct > 0.0001:
                ax.text(x + offset, pct + err + 0.01,
                        f'{pct:.3f}%',
                        ha='center', va='bottom',
                        fontsize=8.5, fontweight='bold',
                        color=C_ANNO)

    # pile-up reference lines
    for arr, colour in [(arr_all, C_REF_ALL), (arr_fv, C_REF_FV)]:
        n_pile = (arr > 1).sum()
        if n_pile > 0:
            ax.axhline(100*n_pile/N, ls='--', lw=1.4,
                       color=colour, alpha=0.6, zorder=2)

    ax.set_xlabel(r'Number of $\nu$ interactions per spill', fontsize=12)
    ax.set_ylabel('Fraction of spills (%)', fontsize=12)
    ax.set_title(title, fontsize=12, pad=8)
    ax.set_xticks(xs)
    ax.set_xticklabels(xlbls, fontsize=11)
    ax.set_xlim(-0.65, 3.65)
    ax.tick_params(axis='x', which='minor', bottom=False, top=False)

    # legend with pile-up rates
    n_all_pile = (arr_all > 1).sum()
    n_fv_pile  = (arr_fv  > 1).sum()
    e_all = binom_err(n_all_pile, N)
    e_fv  = binom_err(n_fv_pile,  N)
    extra = [
        mpatches.Patch(color='none',
            label=f'All pile-up: ({100*n_all_pile/N:.4f} +/- {100*e_all:.4f})%'),
        mpatches.Patch(color='none',
            label=f'FV pile-up:  ({100*n_fv_pile/N:.4f} +/- {100*e_fv:.4f})%'),
        mpatches.Patch(facecolor='none', edgecolor=C_ANNO, hatch='xxx',
            label=r'$\pm 1\sigma$ uncertainty'),
    ]
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles + extra,
              labels + [e.get_label() for e in extra],
              fontsize=9, loc='upper right',
              handlelength=1.4, borderpad=0.8)

    sample_lbl = 'Pilot' if is_pilot else 'Full sample'
    ax.text(0.02, 0.97, f'{sample_lbl}: {N:,} spills',
            transform=ax.transAxes, ha='left', va='top',
            fontsize=9, color=C_ANNO,
            bbox=dict(boxstyle='round,pad=0.3', fc='white',
                      alpha=0.9, ec='lightgrey'))

print('Setup complete.')

---
# Part 1: Pilot Study ($200$ files)

Runs inline -- fast enough to not need chunking.

---

## Study A -- Quick proxy via `truth.index`

In [ ]:
N_PILOT = 200

idx_all = []
n_skip  = 0

print(f'Running Study A over {N_PILOT} files...')
for i, fpath in enumerate(ALL_FILES[:N_PILOT]):
    try:
        tree = open_tree(fpath)
        if tree is None or 'rec.slc.truth.index' not in tree:
            n_skip += 1; continue
        idx = ak.to_numpy(ak.flatten(
            tree['rec.slc.truth.index'].array(library='ak'), axis=None))
        idx_all.append(idx)
    except Exception:
        n_skip += 1
    if (i + 1) % PRINT_EVERY == 0:
        print(f'  [{i+1:>4}/{N_PILOT}]  ok={i+1-n_skip}  '
              f'skipped={n_skip}  '
              f'slices so far={sum(len(x) for x in idx_all):,}')

idx    = np.concatenate(idx_all)
nu_idx = idx[idx >= 0]
n_tot  = len(nu_idx)
n_pile = (nu_idx > 0).sum()
rate_A = n_pile / n_tot if n_tot > 0 else 0
err_A  = binom_err(n_pile, n_tot)

print(f'\n--- Study A results (pilot) ---')
print(f'Files processed           : {N_PILOT - n_skip}')
print(f'Nu-matched slices         : {n_tot:,}')
print(f'truth.index == 0 (1st nu) : {(nu_idx==0).sum():,}  ({100*(nu_idx==0).mean():.4f}%)')
print(f'truth.index == 1 (2nd nu) : {(nu_idx==1).sum():,}  ({100*(nu_idx==1).mean():.4f}%)')
print(f'truth.index  > 0 (proxy)  : {n_pile:,}  ({100*rate_A:.4f} {PM} {100*err_A:.4f})%')
print(f'Collaborator estimate ~1.0% -- pilot gives {100*rate_A:.3f}%')

## Studies B and C -- Truth-level neutrino multiplicity per spill

In [ ]:
nnu_all_spill = []
nnu_fv_spill  = []
n_skip2       = 0

print(f'Running Studies B and C over {N_PILOT} files...')
for i, fpath in enumerate(ALL_FILES[:N_PILOT]):
    try:
        tree = open_tree(fpath)
        if tree is None: n_skip2 += 1; continue
        needed = ['rec.mc.nu.position.x',
                  'rec.mc.nu.position.y',
                  'rec.mc.nu.position.z']
        if not all(b in tree for b in needed): n_skip2 += 1; continue
        x  = tree['rec.mc.nu.position.x'].array(library='ak')
        y  = tree['rec.mc.nu.position.y'].array(library='ak')
        z  = tree['rec.mc.nu.position.z'].array(library='ak')
        fv = in_fv(x, y, z)
        nnu_all_spill.append(ak.to_numpy(ak.num(x,  axis=1)))
        nnu_fv_spill.append( ak.to_numpy(ak.sum(fv, axis=1)))
    except Exception:
        n_skip2 += 1
    if (i + 1) % PRINT_EVERY == 0:
        print(f'  [{i+1:>4}/{N_PILOT}]  ok={i+1-n_skip2}  '
              f'skipped={n_skip2}  '
              f'spills so far={sum(len(x) for x in nnu_all_spill):,}')

nnu_all_p = np.concatenate(nnu_all_spill)
nnu_fv_p  = np.concatenate(nnu_fv_spill)
N_p       = len(nnu_all_p)

print(f'\n--- Studies B and C results (pilot) ---')
print(f'Files processed : {N_PILOT - n_skip2} | Total spills: {N_p:,}')
print()
for label, arr in [('All nu (Study B)', nnu_all_p),
                   ('FV nu  (Study C)', nnu_fv_p)]:
    print(f'{label}:')
    for k in range(4):
        n_k = (arr == k).sum() if k < 3 else (arr >= 3).sum()
        lbl = f'nnu == {k}' if k < 3 else 'nnu >= 3'
        print(f'  {lbl}: {n_k:>7,}  ({100*n_k/N_p:.4f} {PM} {100*binom_err(n_k, N_p):.4f})%')
    n_gt1 = (arr > 1).sum()
    print(f'  pile-up: {n_gt1:>7,}  '
          f'({100*n_gt1/N_p:.4f} {PM} {100*binom_err(n_gt1, N_p):.4f})%')
    print()

## Part $1$ -- Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
make_combined_plot(
    ax, nnu_all_p, nnu_fv_p,
    f'Neutrino multiplicity per spill -- Pilot ($200$ files, ${N_p:,}$ spills)',
    is_pilot=True)
fig.tight_layout()
plt.savefig('pileup_pilot.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part $2$: Full Study -- Chunked approach

The full file list is split into chunks of $1{,}000$ files. Each chunk saves its result as a `.npz` file immediately on completion. This means:
- If the terminal hangs or is interrupted, restart and it skips already-completed chunks automatically
- Multiple terminals can run different chunk ranges in parallel (see instructions below)
- Results are combined in a separate cell once all chunks are done

**To run in parallel across $4$ terminals**, open $4$ terminal sessions and in each run:
```
# terminal 1:  CHUNK_START=0    CHUNK_END=8
# terminal 2:  CHUNK_START=8    CHUNK_END=15
# terminal 3:  CHUNK_START=15   CHUNK_END=22
# terminal 4:  CHUNK_START=22   CHUNK_END=29
```
Set `CHUNK_START` and `CHUNK_END` in the cell below before running.

---

## Run chunks

In [ ]:
# ── Set these per terminal when running in parallel ───────────────────────────
CHUNK_START = 0              # first chunk index to process (inclusive)
CHUNK_END   = len(CHUNKS)    # last chunk index to process (exclusive)
# ─────────────────────────────────────────────────────────────────────────────

n_chunks_done   = 0
n_chunks_skip   = 0
n_chunks_target = CHUNK_END - CHUNK_START

print(f'Processing chunks {CHUNK_START} to {CHUNK_END-1} '
      f'({n_chunks_target} chunks, ~{n_chunks_target * CHUNK_SIZE:,} files)')
print(f'Saving to: {CHUNK_DIR}')
print(f'Chunks already done will be skipped automatically.')
print()

for chunk_idx in range(CHUNK_START, CHUNK_END):
    outfile = f'{CHUNK_DIR}/chunk_{chunk_idx:04d}.npz'

    # skip if already done
    if os.path.exists(outfile):
        n_chunks_skip += 1
        if n_chunks_skip <= 3 or chunk_idx == CHUNK_END - 1:
            print(f'  chunk {chunk_idx:04d} -- already done, skipping')
        elif n_chunks_skip == 4:
            print(f'  (further already-done chunks suppressed...)')
        continue

    chunk    = CHUNKS[chunk_idx]
    idx_c    = []
    nnu_all_c = []
    nnu_fv_c  = []
    n_skip_c  = 0
    n_done_c  = 0

    for i, fpath in enumerate(chunk):
        try:
            tree = open_tree(fpath)
            if tree is None: n_skip_c += 1; continue

            # Study A
            if 'rec.slc.truth.index' in tree:
                idx = ak.to_numpy(ak.flatten(
                    tree['rec.slc.truth.index'].array(library='ak'), axis=None))
                idx_c.append(idx)

            # Studies B and C
            needed = ['rec.mc.nu.position.x',
                      'rec.mc.nu.position.y',
                      'rec.mc.nu.position.z']
            if all(b in tree for b in needed):
                x  = tree['rec.mc.nu.position.x'].array(library='ak')
                y  = tree['rec.mc.nu.position.y'].array(library='ak')
                z  = tree['rec.mc.nu.position.z'].array(library='ak')
                fv = in_fv(x, y, z)
                nnu_all_c.append(ak.to_numpy(ak.num(x,  axis=1)))
                nnu_fv_c.append( ak.to_numpy(ak.sum(fv, axis=1)))

            n_done_c += 1
        except Exception:
            n_skip_c += 1

        if (i + 1) % PRINT_EVERY == 0:
            print(f'  chunk {chunk_idx:04d}  [{i+1:>4}/{len(chunk)}]  '
                  f'ok={n_done_c}  skipped={n_skip_c}')

    # save chunk immediately
    np.savez(outfile,
             idx     = np.concatenate(idx_c)     if idx_c     else np.array([]),
             nnu_all = np.concatenate(nnu_all_c) if nnu_all_c else np.array([]),
             nnu_fv  = np.concatenate(nnu_fv_c)  if nnu_fv_c  else np.array([]),
             n_skip  = np.array([n_skip_c]),
             n_done  = np.array([n_done_c]))

    n_chunks_done += 1
    print(f'  chunk {chunk_idx:04d} saved  '
          f'({n_done_c} ok, {n_skip_c} skipped)  '
          f'[{n_chunks_done}/{n_chunks_target - n_chunks_skip} to go]')

print(f'\nDone. {n_chunks_done} chunks processed, {n_chunks_skip} already existed.')

## Check chunk progress

Run this at any time to see how many chunks are complete.

In [ ]:
done   = [i for i in range(len(CHUNKS))
          if os.path.exists(f'{CHUNK_DIR}/chunk_{i:04d}.npz')]
missing = [i for i in range(len(CHUNKS))
           if not os.path.exists(f'{CHUNK_DIR}/chunk_{i:04d}.npz')]

print(f'Chunks complete : {len(done):>4} / {len(CHUNKS)}')
print(f'Chunks missing  : {len(missing):>4}')
if missing:
    print(f'Missing chunk indices: {missing[:20]}'
          f'{"..." if len(missing) > 20 else ""}')
else:
    print('All chunks complete -- ready to combine!')

## Combine chunks and produce results

Run this cell once all (or enough) chunks are complete. It combines whatever chunks exist and produces the final numbers and plot. Missing chunks are reported but do not prevent the cell from running.

In [ ]:
idx_full   = []
nnu_all_f_list = []
nnu_fv_f_list  = []
n_files_done   = 0
n_files_skip   = 0
n_chunks_loaded = 0

for chunk_idx in range(len(CHUNKS)):
    outfile = f'{CHUNK_DIR}/chunk_{chunk_idx:04d}.npz'
    if not os.path.exists(outfile):
        continue
    data = np.load(outfile)
    if len(data['idx'])     > 0: idx_full.append(data['idx'])
    if len(data['nnu_all']) > 0: nnu_all_f_list.append(data['nnu_all'])
    if len(data['nnu_fv'])  > 0: nnu_fv_f_list.append(data['nnu_fv'])
    n_files_done += int(data['n_done'][0])
    n_files_skip += int(data['n_skip'][0])
    n_chunks_loaded += 1

idx_f     = np.concatenate(idx_full)     if idx_full     else np.array([])
nnu_all_f = np.concatenate(nnu_all_f_list) if nnu_all_f_list else np.array([])
nnu_fv_f  = np.concatenate(nnu_fv_f_list)  if nnu_fv_f_list  else np.array([])
N_spills  = len(nnu_all_f)

nu_idx_f  = idx_f[idx_f >= 0] if len(idx_f) > 0 else np.array([])
n_tot_f   = len(nu_idx_f)
n_pile_f  = (nu_idx_f > 0).sum() if n_tot_f > 0 else 0
rate_Af   = n_pile_f / n_tot_f if n_tot_f > 0 else 0
err_Af    = binom_err(n_pile_f, n_tot_f)

print(f'Chunks loaded   : {n_chunks_loaded} / {len(CHUNKS)}')
print(f'Files processed : {n_files_done:,}')
print(f'Files skipped   : {n_files_skip:,}')
print(f'Total spills    : {N_spills:,}')
print()

# Study A
print(f'Study A (proxy):')
print(f'  Nu-matched slices         : {n_tot_f:,}')
print(f'  truth.index == 0 (1st nu) : {(nu_idx_f==0).sum():,}')
print(f'  truth.index == 1 (2nd nu) : {(nu_idx_f==1).sum():,}')
print(f'  truth.index  > 0          : {n_pile_f:,}  '
      f'({100*rate_Af:.4f} {PM} {100*err_Af:.4f})%')
print()

# Studies B and C
if N_spills > 0:
    for label, arr in [('All nu  (Study B)', nnu_all_f),
                       ('FV nu   (Study C)', nnu_fv_f)]:
        print(f'{label}:')
        for k in range(4):
            n_k = (arr == k).sum() if k < 3 else (arr >= 3).sum()
            lbl = f'nnu == {k}' if k < 3 else 'nnu >= 3'
            print(f'  {lbl}: {n_k:>9,}  '
                  f'({100*n_k/N_spills:.4f} {PM} {100*binom_err(n_k, N_spills):.4f})%')
        n_gt1 = (arr > 1).sum()
        print(f'  pile-up: {n_gt1:>9,}  '
              f'({100*n_gt1/N_spills:.4f} {PM} {100*binom_err(n_gt1, N_spills):.4f})%')
        print()

## Part $2$ -- Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
make_combined_plot(
    ax, nnu_all_f, nnu_fv_f,
    f'Neutrino multiplicity per spill -- {n_chunks_loaded} chunks, '
    f'{n_files_done:,} files, {N_spills:,} spills',
    is_pilot=False)

# overlay Study A proxy
if rate_Af > 0:
    ax.axhline(100*rate_Af, ls=':', lw=1.5, color=C_ANNO, alpha=0.7,
               label=f'Study A proxy: ({100*rate_Af:.4f} +/- {100*err_Af:.4f})%')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, fontsize=9, loc='upper right',
              handlelength=1.4, borderpad=0.8)

fig.tight_layout()
#plt.savefig('pileup_full.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fixed plot: pile-up bins only, Study C conditioned on >= 1 FV nu ──────────

nnu_fv_has1  = nnu_fv_f[nnu_fv_f >= 1]
N_has1       = len(nnu_fv_has1)
n_pile_fv    = (nnu_fv_has1 > 1).sum()
rate_fv_cond = n_pile_fv / N_has1 if N_has1 > 0 else 0
err_fv_cond  = binom_err(n_pile_fv, N_has1)

print(f'Spills with >= 1 FV nu : {N_has1:,}  ({100*N_has1/N_spills:.2f}% of all spills)')
print(f'Pile-up (nnu_fv > 1)   : {n_pile_fv:,}  '
      f'({100*rate_fv_cond:.4f} {PM} {100*err_fv_cond:.4f})%')

xs    = np.array([2, 3])
w     = 0.38
xlbls = ['$2$', r'$\geq 3$']

fig, ax = plt.subplots(figsize=(14, 12), dpi=250)

# Study B -- all nu, denom = all spills
counts_all = [(nnu_all_f == 2).sum(), (nnu_all_f >= 3).sum()]
pcts_all   = [100*c/N_spills for c in counts_all]
errs_all   = [100*binom_err(c, N_spills) for c in counts_all]

ax.bar(xs - w/2, pcts_all, width=w, color=C_ALL,
       edgecolor=C_EDGE, linewidth=0.8,
       label=r'All $\nu$ (Study B -- upper bound)', zorder=3)
for x, pct, err in zip(xs, pcts_all, errs_all):
    if pct > 0 and err > 0:
        ax.add_patch(plt.Rectangle(
            (x - w, pct - err), w, 2*err,
            hatch='xxx', fill=False, edgecolor=C_ANNO, linewidth=0.4, zorder=5))
    ax.text(x - w/2, pct + err + 0.003, f'{pct:.3f}%',
            ha='center', va='bottom', fontsize=9,
            fontweight='bold', color=C_ANNO)

# Study C -- FV nu, conditioned on >= 1 FV nu
counts_fv = [(nnu_fv_has1 == 2).sum(), (nnu_fv_has1 >= 3).sum()]
pcts_fv   = [100*c/N_has1 for c in counts_fv]
errs_fv   = [100*binom_err(c, N_has1) for c in counts_fv]

ax.bar(xs + w/2, pcts_fv, width=w, color=C_FV,
       edgecolor=C_EDGE, linewidth=0.8,
       label=r'FV $\nu$ (Study C -- figure of merit)', zorder=3)
for x, pct, err in zip(xs, pcts_fv, errs_fv):
    if pct > 0 and err > 0:
        ax.add_patch(plt.Rectangle(
            (x, pct - err), w, 2*err,
            hatch='xxx', fill=False, edgecolor=C_ANNO, linewidth=0.4, zorder=5))
    ax.text(x + w/2, pct + err + 0.003, f'{pct:.3f}%',
            ha='center', va='bottom', fontsize=9,
            fontweight='bold', color=C_ANNO)

# pile-up reference lines
ax.axhline(100*(nnu_all_f>1).sum()/N_spills, ls='--', lw=1.4,
           color=C_REF_ALL, alpha=0.6)
ax.axhline(100*rate_fv_cond, ls='--', lw=1.4,
           color=C_REF_FV, alpha=0.6)

extra = [
    mpatches.Patch(color='none',
        label=f'All pile-up: ({100*(nnu_all_f>1).sum()/N_spills:.4f} {PM} '
              f'{100*binom_err((nnu_all_f>1).sum(), N_spills):.4f})%'),
    mpatches.Patch(color='none',
        label=f'FV pile-up (conditioned): ({100*rate_fv_cond:.4f} {PM} '
              f'{100*err_fv_cond:.4f})%'),
    mpatches.Patch(facecolor='none', edgecolor=C_ANNO, hatch='xxx',
        label=r'$\pm 1\sigma$ uncertainty'),
]
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles + extra, labels + [e.get_label() for e in extra],
          fontsize=9, loc='upper right', handlelength=1.4)

ax.set_xlabel(r'Number of $\nu$ interactions per spill', fontsize=12)
ax.set_ylabel('Fraction of spills (%)', fontsize=12)
ax.set_title(
    f'Neutrino pile-up -- {n_chunks_loaded} chunks, {n_files_done:,} files, '
    f'{N_spills:,} spills\n'
    r'Study C conditioned on spills with $\geq 1$ FV $\nu$',
    fontsize=12)
ax.set_xticks(xs)
ax.set_xticklabels(xlbls, fontsize=11)
ax.set_xlim(1.4, 3.6)
ax.tick_params(axis='x', which='minor', bottom=False, top=False)
ax.text(0.02, 0.97, f'Full sample: {N_spills:,} spills',
        transform=ax.transAxes, ha='left', va='top', fontsize=9, color=C_ANNO,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.9, ec='lightgrey'))

fig.tight_layout()
#plt.savefig('pileup_full_conditioned.png', dpi=150, bbox_inches='tight')
plt.show()

## Final Summary

In [ ]:
p_all_p = (nnu_all_p>1).mean(); e_all_p = binom_err((nnu_all_p>1).sum(), N_p)
p_fv_p  = (nnu_fv_p >1).mean(); e_fv_p  = binom_err((nnu_fv_p >1).sum(), N_p)
p_all_f = (nnu_all_f>1).mean() if N_spills > 0 else 0
e_all_f = binom_err((nnu_all_f>1).sum(), N_spills) if N_spills > 0 else 0
p_fv_f  = (nnu_fv_f >1).mean() if N_spills > 0 else 0
e_fv_f  = binom_err((nnu_fv_f >1).sum(),  N_spills) if N_spills > 0 else 0

w = 60
print('=' * w)
print('NEUTRINO PILE-UP SUMMARY -- ICARUS BNB MC')
print('=' * w)
print(f'  {"":<36} {"Pilot":>8}  {f"{n_chunks_loaded} chunks":>12}')
print(f'  {"Spills analysed":<36} {N_p:>8,}  {N_spills:>12,}')
print('-' * w)
print(f'  Study A  proxy (truth.index > 0)')
print(f'  {"rate":<36} {100*rate_A:>7.4f}%  '
      f'({100*rate_Af:.4f} {PM} {100*err_Af:.4f})%')
print()
print(f'  Study B  all nu > 1 per spill  [upper bound]')
print(f'  {"rate":<36} {100*p_all_p:>7.4f}%  '
      f'({100*p_all_f:.4f} {PM} {100*e_all_f:.4f})%')
print()
print(f'  Study C  FV nu > 1 per spill   [main FOM]')
print(f'  {"rate":<36} {100*p_fv_p:>7.4f}%  '
      f'({100*p_fv_f:.4f} {PM} {100*e_fv_f:.4f})%')
print()
print('=' * w)
print()
print('INTERPRETATION')
print('-' * w)
print(f'  Main FOM (Study C): {100*p_fv_f:.4f}% of spills have > 1 FV neutrino.')
print(f'  Study A proxy consistent at {100*rate_Af:.4f}%.')
print()
if p_fv_f < 0.02:
    print('  CONCLUSION: FV pile-up rate is well below 1%.')
    print('  SBND-style pile-up handling is likely overkill for ICARUS BNB.')
else:
    print('  CONCLUSION: FV pile-up rate is non-negligible.')
    print('  SBND-style pile-up handling may be worth pursuing.')
print('=' * w)